# Imports + Loading Model

In [3]:
import os
import sys

# Clone the V-JEPA 2 repository if it doesn't exist
repo_dir = 'vjepa2'
if not os.path.exists(repo_dir):
    !git clone https://github.com/facebookresearch/vjepa2.git
    # Change to the cloned directory to install requirements
    # %cd vjepa2
    # !pip install -r requirements.txt
    # %cd .. # Change back to the original directory

# Add the cloned repository to the system path
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

In [11]:
import json
import os
import subprocess

import numpy as np
import torch
import torch.nn.functional as F
from decord import VideoReader
from transformers import AutoVideoProcessor, AutoModel

import src.datasets.utils.video.transforms as video_transforms
import src.datasets.utils.video.volume_transforms as volume_transforms
from src.models.attentive_pooler import AttentiveClassifier
from src.models.vision_transformer import vit_large_rope
from app.vjepa_2_1.models.vision_transformer import vit_base, vit_large

IMAGENET_DEFAULT_MEAN = (0.485, 0.456, 0.406)
IMAGENET_DEFAULT_STD = (0.229, 0.224, 0.225)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device: ', device)

def load_pretrained_vjepa_pt_weights(model, pretrained_weights):
    # Load weights of the VJEPA2 encoder
    # The PyTorch state_dict is already preprocessed to have the right key names
    pretrained_dict = torch.load(pretrained_weights, weights_only=True, map_location=device)["encoder"]
    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
    pretrained_dict = {k.replace("backbone.", ""): v for k, v in pretrained_dict.items()}
    msg = model.load_state_dict(pretrained_dict, strict=False)
    print("Pretrained weights found at {} and loaded with msg: {}".format(pretrained_weights, msg))

def build_pt_video_transform(img_size):
    short_side_size = int(256.0 / 224 * img_size)
    # Eval transform has no random cropping nor flip
    eval_transform = video_transforms.Compose(
        [
            video_transforms.Resize(short_side_size, interpolation="bilinear"),
            video_transforms.CenterCrop(size=(img_size, img_size)),
            volume_transforms.ClipToTensor(),
            video_transforms.Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD),
        ]
    )
    return eval_transform



def get_video(path):
    vr = VideoReader(path)
    last_frame_idx = min(128, len(vr) - 1)
    # choosing some frames here, you can define more complex sampling strategy
    frame_idx = np.arange(0, last_frame_idx, 2)
    video = vr.get_batch(frame_idx).asnumpy()
    return video

def forward_vjepa_video(model_pt, pt_transform, path):
  # Run a sample inference with VJEPA
  with torch.inference_mode():
      # Read and pre-process the image
      video = get_video(path)  # T x H x W x C
      video = torch.from_numpy(video).permute(0, 3, 1, 2)  # T x C x H x W
      x_pt = pt_transform(video).cuda().unsqueeze(0)
      # Extract the patch-wise features from the last layer
      out_patch_features_pt = model_pt(x_pt)

  return out_patch_features_pt

device:  cuda


In [ ]:
import torch

img_size = 384
# pt_model_path = '/content/drive/MyDrive/IDP/models/vjepa2_1_vitb_dist_vitG_384.pt' # base
# pt_model_path = '/content/drive/MyDrive/IDP/models/vjepa2_1_vitl_dist_vitG_384.pt' # large
# pt_model_path = '/content/drive/MyDrive/IDP/models/vitl.pt' # large2.0
pt_model_path = './checkpoints/vitl.pt' # large2.0

# Build PyTorch preprocessing transform
pt_video_transform = build_pt_video_transform(img_size=img_size)

# Initialize the PyTorch model, load pretrained weights
# model = vit_base(img_size=(img_size, img_size), num_frames=64, use_rope=True) # base
# model = vit_large(img_size=(img_size, img_size), num_frames=64, use_rope=True) # 2.1 large
model = vit_large_rope(img_size=(img_size, img_size), num_frames=16) # 2.0 large
model.cuda().eval()
load_pretrained_vjepa_pt_weights(model, pt_model_path)
preprocessor = build_pt_video_transform(img_size=img_size)

IsADirectoryError: [Errno 21] Is a directory: './'

# Sample Data loading + prediction

In [ ]:
!ffmpeg -i "/content/drive/MyDrive/IDP/ss2/Letting something roll up a slanted surface, so it rolls back down/134018.webm" \
        "/content/vid.mp4" -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
sample_video_path = "sample_video.mp4"
# Download the video if not yet downloaded to local path
if not os.path.exists(sample_video_path):
    video_url = "https://huggingface.co/datasets/nateraw/kinetics-mini/resolve/main/val/bowling/-WH-lxmGJVY_000005_000015.mp4"
    command = ["wget", video_url, "-O", sample_video_path]
    subprocess.run(command)
    print("Downloading video")

# Download SSV2 classes if not already present
ssv2_classes_path = "ssv2_classes.json"
if not os.path.exists(ssv2_classes_path):
    command = [
        "wget",
        "https://huggingface.co/datasets/huggingface/label-files/resolve/d79675f2d50a7b1ecf98923d42c30526a51818e2/"
        "something-something-v2-id2label.json",
        "-O",
        "ssv2_classes.json",
    ]
    subprocess.run(command)
    print("Downloading SSV2 classes")

In [ ]:
sample_video_path = "vid.mp4"
# sample_video_path = "sample_video.mp4"
with torch.no_grad():
  output = forward_vjepa_video(model, preprocessor, sample_video_path)

/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


In [ ]:
!wget https://dl.fbaipublicfiles.com/vjepa2/evals/ssv2-vitl-16x2x3.pt -P /content/

--2026-05-15 09:11:46--  https://dl.fbaipublicfiles.com/vjepa2/evals/ssv2-vitl-16x2x3.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.249.182.62, 13.249.182.39, 13.249.182.33, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.249.182.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 198078572 (189M) [application/vnd.snesdev-page-table]
Saving to: ‘/content/ssv2-vitl-16x2x3.pt’

ssv2-vitl-16x2x3.pt 100%[===================>] 188.90M  27.0MB/s    in 7.3s    

2026-05-15 09:11:54 (25.8 MB/s) - ‘/content/ssv2-vitl-16x2x3.pt’ saved [198078572/198078572]



In [ ]:
def load_pretrained_vjepa_classifier_weights(model, pretrained_weights):
    # Load weights of the VJEPA2 classifier
    # The PyTorch state_dict is already preprocessed to have the right key names
    pretrained_dict = torch.load(pretrained_weights, weights_only=True, map_location=device)["classifiers"][0]
    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
    msg = model.load_state_dict(pretrained_dict, strict=False)
    print("Pretrained weights found at {} and loaded with msg: {}".format(pretrained_weights, msg))

def get_vjepa_video_classification_results(classifier, out_patch_features_pt):
    SOMETHING_SOMETHING_V2_CLASSES = json.load(open("ssv2_classes.json", "r"))

    with torch.inference_mode():
        out_classifier = classifier(out_patch_features_pt)

    print(f"Classifier output shape: {out_classifier.shape}")

    print("Top 5 predicted class names:")
    top5_indices = out_classifier.topk(5).indices[0]
    top5_probs = F.softmax(out_classifier.topk(5).values[0], dim=0) * 100.0  # convert to percentage
    for idx, prob in zip(top5_indices, top5_probs):
        str_idx = str(idx.item())
        print(f"{SOMETHING_SOMETHING_V2_CLASSES[str_idx]} ({prob}%)")

    return

In [ ]:
# Initialize the classifier
classifier_model_path = "/content/ssv2-vitl-16x2x3.pt"
classifier = (
    AttentiveClassifier(embed_dim=model.embed_dim, num_heads=16, depth=4, num_classes=174).to(device).eval()
)
load_pretrained_vjepa_classifier_weights(classifier, classifier_model_path)

# Get classification results
get_vjepa_video_classification_results(classifier, output)

Pretrained weights found at /content/ssv2-vitl-16x2x3.pt and loaded with msg: <All keys matched successfully>
Classifier output shape: torch.Size([1, 174])
Top 5 predicted class names:
Pushing [something] off of [something] (58.75494384765625%)
Poking [something] so that it falls over (22.340112686157227%)
Moving [something] across a surface until it falls down (11.19368839263916%)
Poking a stack of [something] so the stack collapses (5.355512619018555%)
Pushing [something] so that it falls off the table (2.355743646621704%)
